# Per-fold Per-class Correct/Incorrect — Swin-L vs EffNet-B7 vs EffNetV2-L

- 각 fold에 대해 세 모델의 per-class 정/오답 막대그래프를 **겹쳐서** 비교합니다.
- 소스: 각 run_dir의 per_class_best.json (없으면 latest).
- 저장: reports/summary/<timestamp>/per_fold_compare_fold{f}_correct.png / _incorrect.png
- 비고: EfficientNetV2-L은 일부 fold만 있을 수 있어, 해당 fold에서만 겹칩니다.


In [1]:
import os, glob, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')
out_dir = Path('reports/summary') / time.strftime('%Y%m%d-%H%M%S')
out_dir.mkdir(parents=True, exist_ok=True)
print('Artifacts will be saved to:', out_dir)


In [2]:
def identify_model(run_name: str):
    n = run_name.lower()
    if 'tf_efficientnet_b7_ns' in n: return 'effnet_b7'
    if 'swin_large_patch4_window12' in n: return 'swin_large'
    if 'efficientnetv2_l' in n: return 'effnetv2_l'
    return 'other'

def load_fold(run_dir: Path):
    import yaml
    try:
        cfg = yaml.safe_load((run_dir/'config.yaml').read_text())
        return int(cfg['split']['fold_index'])
    except Exception:
        return None

def load_per_class(run_dir: Path):
    for name in ['per_class_best.json','per_class_latest.json']:
        p = run_dir / name
        if p.exists():
            try:
                return json.loads(p.read_text())
            except Exception:
                pass
    return None

def collect_runs():
    data = []
    for p in sorted(glob.glob('outputs/runs/*')):
        rd = Path(p)
        if not rd.is_dir(): continue
        model = identify_model(rd.name)
        if model=='other': continue
        fold = load_fold(rd)
        pc = load_per_class(rd)
        if fold is None or pc is None: continue
        data.append({'run_dir': rd.name, 'model': model, 'fold': fold, 'pc': pc})
    return pd.DataFrame(data)

runs_df = collect_runs()
print('Found usable runs:', len(runs_df))
runs_df.head()


In [3]:
def plot_fold_compare(df: pd.DataFrame, fold: int, out_dir: Path):
    sub = df[df['fold']==fold]
    if sub.empty:
        print(f'Fold {fold}: no data. skip')
        return
    models = ['swin_large','effnet_b7','effnetv2_l']
    label_map = {'swin_large':'Swin-L','effnet_b7':'EffNet-B7','effnetv2_l':'EffNetV2-L'}
    colors = {'swin_large':'#1f78b4','effnet_b7':'#33a02c','effnetv2_l':'#e31a1c'}
    # Gather per-class counts
    series = {}
    n_cls = None
    for m in models:
        row = sub[sub['model']==m]
        if row.empty: continue
        pc = row.iloc[0]['pc']
        tot = pc.get('per_class_total') or []
        cor = pc.get('per_class_correct') or []
        if not tot or not cor: continue
        inc = [int(t)-int(c) for t,c in zip(tot, cor)]
        n_cls = len(tot)
        series[m] = {'total': tot, 'correct': cor, 'incorrect': inc}
    if not series:
        print(f'Fold {fold}: missing per-class series. skip')
        return
    classes = np.arange(n_cls)
    width = 0.25
    # Correct overlay
    plt.figure(figsize=(12,4))
    offset = -width
    for m in models:
        if m not in series: continue
        vals = series[m]['correct']
        plt.bar(classes + offset, vals, width=width, label=label_map[m], color=colors[m])
        offset += width
    plt.title(f'Fold {fold} — Per-class Correct (overlay)')
    plt.xlabel('Class'); plt.ylabel('Count'); plt.legend(); plt.tight_layout()
    plt.savefig(out_dir / f'per_fold_compare_fold{fold}_correct.png', dpi=150); plt.close()
    # Incorrect overlay
    plt.figure(figsize=(12,4))
    offset = -width
    for m in models:
        if m not in series: continue
        vals = series[m]['incorrect']
        plt.bar(classes + offset, vals, width=width, label=label_map[m], color=colors[m])
        offset += width
    plt.title(f'Fold {fold} — Per-class Incorrect (overlay)')
    plt.xlabel('Class'); plt.ylabel('Count'); plt.legend(); plt.tight_layout()
    plt.savefig(out_dir / f'per_fold_compare_fold{fold}_incorrect.png', dpi=150); plt.close()
    print('Saved fold', fold)

for f in sorted(runs_df['fold'].dropna().unique().astype(int)):
    plot_fold_compare(runs_df, f, out_dir)
print('All done. Artifacts at:', out_dir)
